# GitHub Issue Triage Agent -- Colab / Kaggle / Binder notebook

This notebook is a self-contained, notebook-friendly port of the real
[`examples/github-issue-triage-agent/triage.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/github-issue-triage-agent/triage.py)
script from the course's Real-World Projects lesson
([`docs/projects/github-issue-triage-agent/index.md`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/docs/projects/github-issue-triage-agent/index.md)).

It fetches **OPEN** issues from a real public GitHub repo (defaults to
[`psf/requests`](https://github.com/psf/requests), same as the script), asks a free-tier LLM
to suggest a triage label and a one-sentence rationale for each one, and prints a readable
report -- for a human maintainer to read and decide on.

**This only drafts suggestions. It never applies a label, comment, or any other change to a
real issue** -- it never calls a GitHub endpoint capable of writing anything.

By default this uses [GitHub Models](https://github.com/marketplace/models) -- free, and no
separate signup if you already have a GitHub account. You only need **one** API key: a GitHub
personal access token with the `models: read` scope, from
[github.com/settings/tokens](https://github.com/settings/tokens).

In [ ]:
!pip install -q requests openai

## 1. Provide your free-tier LLM API key

Paste a GitHub personal access token with the `models: read` scope (free, no separate
signup -- see [github.com/settings/tokens](https://github.com/settings/tokens)). It is read
with `getpass` so it never gets echoed to the notebook output or saved in this file.

This is **different** from the optional GitHub REST API token below: this key authenticates
calls to the *LLM*, not to the GitHub issues API.

In [ ]:
from getpass import getpass
import os

os.environ["GITHUB_TOKEN"] = getpass("GitHub personal access token (models: read scope): ")

## 2. (Optional) A separate GitHub token to raise the issues API rate limit

Fetching issues uses GitHub's REST API, which allows unauthenticated requests for public
repos -- but caps them at 60 requests/hour. That's plenty for one run of this notebook (it
makes one request total to list issues), so **feel free to leave this blank and just run the
next cell**. Only fill it in if you're re-running this a lot and hit that limit -- any GitHub
personal access token works, no scopes needed for public repo reads, and it can be the same
token as above.

In [ ]:
github_api_token = getpass("Optional GitHub token for higher REST API rate limits (press Enter to skip): ") or None

## 3. Fetch open issues from a real public repo

Same logic as `fetch_open_issues()` in `triage.py`: pulls OPEN issues from GitHub's REST API,
sorted by most recently updated, and filters out pull requests (a PR *is* an issue internally,
so it shows up in this endpoint too, but lacks a `pull_request` key on true issues).

In [ ]:
import requests

GITHUB_API_URL = "https://api.github.com"
MAX_BODY_CHARS = 2000  # keep each issue's body well inside any model's context window
LABEL_CHOICES = ["bug", "feature", "question", "docs", "duplicate-looking", "other"]


def fetch_open_issues(owner: str, repo: str, limit: int = 10, token: str | None = None) -> list[dict]:
    """Fetch up to `limit` OPEN issues from a public GitHub repo."""
    headers = {"Accept": "application/vnd.github+json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"

    response = requests.get(
        f"{GITHUB_API_URL}/repos/{owner}/{repo}/issues",
        params={"state": "open", "per_page": min(limit, 100), "sort": "updated"},
        headers=headers,
        timeout=10,
    )
    response.raise_for_status()
    issues = [item for item in response.json() if "pull_request" not in item]
    return issues[:limit]


GITHUB_REPO = "psf/requests"  # same default as triage.py -- set to any owner/repo you like
owner, repo = GITHUB_REPO.split("/", 1)

issues = fetch_open_issues(owner, repo, limit=5, token=github_api_token)
print(f"Fetched {len(issues)} open issue(s) from {owner}/{repo}")

## 4. Build a triage-suggestion prompt per issue

Same logic as `build_triage_prompt()` / `parse_triage_reply()` in `triage.py`: ask the model
to pick exactly one label from a fixed list and give a one-sentence rationale, in a strict
two-line reply format that's easy to parse back out.

In [ ]:
def build_triage_prompt(issue: dict) -> str:
    title = issue.get("title") or "(no title)"
    body = (issue.get("body") or "(no description provided)")[:MAX_BODY_CHARS]

    return (
        "You are drafting a SUGGESTION for a human maintainer triaging a GitHub "
        "issue. You are not applying anything -- your output will be reviewed by "
        "a person before any label is added.\n\n"
        f"Choose exactly one label from this list: {', '.join(LABEL_CHOICES)}.\n\n"
        f"Issue title: {title}\n"
        f"Issue body:\n{body}\n\n"
        "Reply in exactly this two-line format, nothing else:\n"
        "Label: <one label from the list>\n"
        "Rationale: <one sentence explaining the suggested label and its priority>"
    )


def parse_triage_reply(reply: str) -> dict:
    label, rationale = "other", reply.strip()
    for line in reply.splitlines():
        if line.lower().startswith("label:"):
            candidate = line.split(":", 1)[1].strip().lower()
            label = candidate if candidate in LABEL_CHOICES else candidate or "other"
        elif line.lower().startswith("rationale:"):
            rationale = line.split(":", 1)[1].strip()
    return {"label": label, "rationale": rationale}

## 5. Call the LLM (GitHub Models, free tier) for each issue

Same OpenAI-compatible call pattern as `_build_github_caller()` in `triage.py` -- GitHub
Models exposes an OpenAI-compatible endpoint, so the regular `openai` package works against it
by pointing `base_url` at `https://models.github.ai/inference`.

In [ ]:
import time
from openai import OpenAI

client = OpenAI(api_key=os.environ["GITHUB_TOKEN"], base_url="https://models.github.ai/inference")


def call_llm(prompt: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,  # a triage suggestion should be consistent, not creative
    )
    return response.choices[0].message.content or ""


def suggest_triage(issue: dict, call_llm) -> dict:
    """Get one suggested label + rationale for one issue. Never touches the real issue."""
    prompt = build_triage_prompt(issue)
    reply = call_llm(prompt)
    return parse_triage_reply(reply)


suggestions = []
for issue in issues:
    suggestions.append(suggest_triage(issue, call_llm))
    time.sleep(0.5)  # a small, deliberate gap between LLM calls

## 6. Print the triage report

Same as `print_triage_report()` in `triage.py`: every suggestion below is a DRAFT for a human
maintainer to review -- nothing here has been applied to the real issues on GitHub.

In [ ]:
def print_triage_report(owner: str, repo: str, issues: list[dict], suggestions: list[dict]) -> None:
    print("=" * 72)
    print(f"Triage suggestions for {owner}/{repo} -- {len(issues)} open issue(s)")
    print("These are DRAFT suggestions. Review each one before applying any label.")
    print("=" * 72)
    for issue, suggestion in zip(issues, suggestions):
        print(f"\n#{issue['number']}: {issue['title']}")
        print(f"  {issue['html_url']}")
        print(f"  Suggested label: {suggestion['label']}")
        print(f"  Rationale:       {suggestion['rationale']}")


print_triage_report(owner, repo, issues, suggestions)